In [25]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import StratifiedKFold

In [26]:
df_train = pd.read_csv("data/train.csv")
df_test = pd.read_csv('data/test.csv')

In [27]:
correlation_matrix = df_train.corr(numeric_only=True)
print(correlation_matrix)

           target     var_0     var_1     var_2     var_3     var_4     var_5  \
target   1.000000  0.052390  0.050343  0.055870  0.011055  0.010915  0.030979   
var_0    0.052390  1.000000 -0.000544  0.006573  0.003801  0.001326  0.003046   
var_1    0.050343 -0.000544  1.000000  0.003980  0.000010  0.000303 -0.000902   
var_2    0.055870  0.006573  0.003980  1.000000  0.001001  0.000723  0.001569   
var_3    0.011055  0.003801  0.000010  0.001001  1.000000 -0.000322  0.003253   
...           ...       ...       ...       ...       ...       ...       ...   
var_195  0.028285  0.002073 -0.000785 -0.001070  0.001206  0.003706 -0.001274   
var_196  0.023608  0.004386 -0.000377  0.003952 -0.002800  0.000513  0.002880   
var_197 -0.035303 -0.000753 -0.004157  0.001078  0.001164 -0.000046 -0.000535   
var_198 -0.053000 -0.005776 -0.004861 -0.000877 -0.001651 -0.001821 -0.000953   
var_199  0.025434  0.003850  0.002287  0.003855  0.000506 -0.000786  0.002767   

            var_6     var_7

In [28]:
duplicates_count = df_train.duplicated().sum()
print(f"Количество дубликатов: {duplicates_count}")

Количество дубликатов: 0


In [29]:
targets = df_train['target']
features = df_train.drop(columns=['target', 'ID_code'])
test_features = df_test.drop(columns=['ID_code'])

In [30]:
def add_features(features):
    features['avg_row'] = features.mean(axis=1)
    features['max_row'] = features.max(axis=1)
    features['min_row'] = features.min(axis=1)

In [31]:
add_features(features)
add_features(test_features)

In [32]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

preds = []
all_val_scores = []
all_train_scores = []
test_predictions = np.zeros(len(test_features))

for fold, (train_idx, val_idx) in enumerate(skf.split(features, targets)):
    X_train, y_train = features.iloc[train_idx], targets.iloc[train_idx]
    X_val, y_val = features.iloc[val_idx], targets.iloc[val_idx]

    model = lgb.LGBMClassifier(
            n_estimators=2000,
            learning_rate=0.05,
            max_depth=3,
            random_state=42,
            n_jobs=-1,
            verbose=-1
        )

    model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)], # Данные для проверки на каждом шаге
            callbacks=[
                lgb.early_stopping(stopping_rounds=50, verbose=False),
                lgb.log_evaluation(period=0) # Отключаем лишний спам логов в консоль
            ]
        )

    preds_train = model.predict_proba(X_train)[:, 1]
    roc_auc_train = roc_auc_score(y_train, preds_train)
    all_train_scores.append(roc_auc_train)

    preds_val = model.predict_proba(X_val)[:, 1]
    roc_auc_val = roc_auc_score(y_val, preds_val)
    all_val_scores.append(roc_auc_val)

    print(f"Фолд {fold + 1} | Train AUC: {roc_auc_train:.4f} | Val AUC: {roc_auc_val:.4f}")

    test_predictions += model.predict_proba(test_features)[:, 1] / skf.n_splits

print(f"Средний Train AUC: {sum(all_train_scores) / len(all_train_scores):.4f}")
print(f"Средний Val AUC:   {sum(all_val_scores) / len(all_val_scores):.4f}")

Фолд 1 | Train AUC: 0.9510 | Val AUC: 0.8954
Фолд 2 | Train AUC: 0.9505 | Val AUC: 0.8961
Фолд 3 | Train AUC: 0.9517 | Val AUC: 0.8891
Фолд 4 | Train AUC: 0.9509 | Val AUC: 0.8944
Фолд 5 | Train AUC: 0.9505 | Val AUC: 0.8962
Средний Train AUC: 0.9509
Средний Val AUC:   0.8942


In [34]:
submission = pd.DataFrame({
    'ID_code': df_test['ID_code'],
    'target': test_predictions
})

submission.to_csv('results/my_lightgbm_submission.csv', index=False)
print("Файл my_lightgbm_submission.csv успешно сохранен!")

Файл my_lightgbm_submission.csv успешно сохранен!
